In [1]:
import pandas as pd

In [3]:
roll_number = "1024160070"
fixed_entries = [
{"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
"keywords": "fee cost price charge", "category": "billing"},
{"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
"keywords": "password reset login", "category": "account"},
{"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
"keywords": "hours timing open time", "category": "general"},
{"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
"keywords": "pay payment upi fee", "category": "billing"},
]

In [4]:
categories = ["billing", "account", "general"]


In [5]:
personalized_templates = {
    "billing": [
        {
            "question": "how can i download my payment receipt",
            "answer": "Open Payment History and select the transaction to download its receipt.",
            "keywords": "receipt payment invoice",
            "category": "billing"
        },
        {
            "question": "when will my payment be updated",
            "answer": "Payments are usually updated within 24 hours after successful processing.",
            "keywords": "payment pending update",
            "category": "billing"
        }
    ],
    "account": [
        {
            "question": "how do i update my registered mobile number",
            "answer": "Go to Profile Settings and select Update Mobile Number.",
            "keywords": "mobile update profile",
            "category": "account"
        },
        {
            "question": "how can i change my email address",
            "answer": "Go to Account Settings, edit your email address, and verify it using OTP.",
            "keywords": "email change account",
            "category": "account"
        }
    ],
    "general": [
        {
            "question": "where can i find customer support",
            "answer": "You can contact customer support through the Help and Support section.",
            "keywords": "support help contact",
            "category": "general"
        },
        {
            "question": "how do i check service availability",
            "answer": "Service availability can be checked on the official website or support portal.",
            "keywords": "service availability status",
            "category": "general"
        }
    ]
}


In [6]:
digits = [int(char) for char in roll_number if char.isdigit()]
last_two_digits = digits[-2:]

In [7]:
personalized_entries = []

for i, digit in enumerate(last_two_digits):
    category = categories[digit % 3]
    entry = personalized_templates[category][i].copy()
    personalized_entries.append(entry)

all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)

print("\nQ1: Final 6-row FAQ DataFrame")
print(df.to_string(index=False))


Q1: Final 6-row FAQ DataFrame
                                   question                                                                    answer               keywords category
                     what is the annual fee                                                 The annual fee is Rs 500.  fee cost price charge  billing
                      how to reset password                                          Go to Settings > Reset Password.   password reset login  account
                what are your working hours                                                 We are open 9 AM to 5 PM. hours timing open time  general
                      how can i pay the fee                                You can pay via UPI, card, or net banking.    pay payment upi fee  billing
how do i update my registered mobile number                   Go to Profile Settings and select Update Mobile Number.  mobile update profile  account
            when will my payment be updated Payments are usually upda

#Q2: Score Matching FAQs


In [8]:
def score_faqs(query, df):
    query_words = set(query.lower().split())
    results = []
    for index, row in df.iterrows():
        searchable_text = (row["question"] + " " + row["keywords"]).lower()
        searchable_words = set(searchable_text.split())
        score = len(query_words.intersection(searchable_words))
        if score > 0:
          results.append({
              "question" : row["question"],
              "answer" : row["answer"],
              "keywords": row["keywords"],
              "category": row["category"],
              "confidence_score" : score
          })
    result_df = pd.DataFrame(results)
    if not result_df.empty:
      result_df = result_df.sort_values(
        by="confidence_score",
        ascending=False
      ).reset_index(drop=True)

    return result_df



In [9]:
print("\nQ2: Query Results for 'password reset'")
q2_result = score_faqs("password reset", df)
print(q2_result.to_string(index=False))


Q2: Query Results for 'password reset'
             question                           answer             keywords category  confidence_score
how to reset password Go to Settings > Reset Password. password reset login  account                 2


#Q3: Return FAQs from the Same Category

In [11]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()][
        ["question", "answer", "keywords", "category"]
    ]

In [12]:
personalized_category = personalized_entries[0]["category"]

print(f"\nQ3: Questions in '{personalized_category}' category")
print(same_category(personalized_category, df).to_string(index=False))


Q3: Questions in 'account' category
                                   question                                                  answer              keywords category
                      how to reset password                        Go to Settings > Reset Password.  password reset login  account
how do i update my registered mobile number Go to Profile Settings and select Update Mobile Number. mobile update profile  account


# Q4: Add New Keyword and Save CSV


In [14]:
entry_index = 0
new_keyword = input(
    f"\nQ4: Enter a new keyword for: '{df.loc[entry_index, 'question']}': "
).strip().lower()



Q4: Enter a new keyword for: 'what is the annual fee': AnnualFees


In [16]:
if new_keyword:
    df.loc[entry_index, "keywords"] += " " + new_keyword
filename = f"{roll_number}_faq_data.csv"
df.to_csv(filename, index=False)
print(f"\nUpdated DataFrame saved successfully as: {filename}")
print(df.to_string(index=False))



Updated DataFrame saved successfully as: 1024160070_faq_data.csv
                                   question                                                                    answer                                    keywords category
                     what is the annual fee                                                 The annual fee is Rs 500. fee cost price charge annualfees annualfees  billing
                      how to reset password                                          Go to Settings > Reset Password.                        password reset login  account
                what are your working hours                                                 We are open 9 AM to 5 PM.                      hours timing open time  general
                      how can i pay the fee                                You can pay via UPI, card, or net banking.                         pay payment upi fee  billing
how do i update my registered mobile number                   Go to Profile Set

# Q5: Count FAQs in Each Category Using groupby


In [17]:
category_count = df.groupby("category").size().reset_index(name="faq_count")


In [18]:
print("\nQ5:Count FAQ per each category:\n")
print(category_count.to_string(index=False))


Q5:Count FAQ per each category:

category  faq_count
 account          2
 billing          3
 general          1


#  Q6: Modify your Q2 scoring function

In [19]:
def best_faq_matches(query, df):
  scored_results = score_faqs(query, df)
  if scored_results.empty:
      print(f"\nNo match found for query: '{query}'")
      return scored_results
  highest_score = scored_results["confidence_score"].max()
  top_matches = scored_results[
    scored_results["confidence_score"] == highest_score
  ].reset_index(drop=True)
  print(f"\nQuery: '{query}'")
  print(f"Highest confidence score: {highest_score}")
  if len(top_matches) > 1:
    print("Tie found.All equally have the best FAQs are: ")
  else:
    print("One matching the best FAQ is: ")
  print(top_matches.to_string(index=False))
  return top_matches


In [20]:
print("\nQ6(a): Tie Demonstration")
best_faq_matches("fee", df)

print("\nQ6(b): Non-Tie Demonstration")
best_faq_matches("password reset", df)


Q6(a): Tie Demonstration

Query: 'fee'
Highest confidence score: 1
Tie found.All equally have the best FAQs are: 
              question                                     answer                                    keywords category  confidence_score
what is the annual fee                  The annual fee is Rs 500. fee cost price charge annualfees annualfees  billing                 1
 how can i pay the fee You can pay via UPI, card, or net banking.                         pay payment upi fee  billing                 1

Q6(b): Non-Tie Demonstration

Query: 'password reset'
Highest confidence score: 2
One matching the best FAQ is: 
             question                           answer             keywords category  confidence_score
how to reset password Go to Settings > Reset Password. password reset login  account                 2


,question,answer,keywords,category,confidence_score
0,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
